## Resumen de insumos por gráfico

Este notebook intenta cargar automáticamente los CSV desde `data/csv/` y usa alias de columnas para tolerar esquemas distintos.

| # | Gráfico | CSV esperado (por defecto) | Columnas mínimas |
|---|---|---|---|
| 1 | Barras de targets RIPE Atlas | `data/csv/ripe_atlas/targets_summary.csv` (o `targets.csv`) | Formato ancho: columnas equivalentes a `prefijos_chile`, `targets_generados`, `con_respuesta`, `sin_respuesta`; **o** formato largo: `categoria` + `cantidad` |
| 2 | Distribución geográfica de probes chilenos | `data/csv/ripe_atlas/probes.csv` | Para scatter: `lat` + `lon` (opcional `country`); para barras: `region` o `city` |
| 3 | Cobertura de atributos PeeringDB | `data/csv/merged/nodes.csv` (fallback `bgp/ripe`) | `has_peeringdb` **o** columnas de atributos tipo `info_*`, `policy_*`, `ix_count`, `fac_count`, `org_id` |
| 4 | Tipos de ASNs sin atributos | `data/csv/merged/nodes.csv` (+ opcional `merged/edges.csv`) | `as_type`; si no existe, `degree_total` o `in_degree`+`out_degree` |
| 5a | Nodos por fuente | `bgp/nodes.csv`, `ripe_atlas/nodes.csv`, `merged/nodes.csv` | Cualquier estructura válida de nodos (se cuenta filas) |
| 5b | Aristas por fuente | `bgp/edges.csv`, `ripe_atlas/edges.csv`, `merged/edges.csv` | Cualquier estructura válida de aristas (se cuenta filas) |
| 6a | Solapamiento ASNs | `bgp/nodes.csv`, `ripe_atlas/nodes.csv` | `asn` |
| 6b | Solapamiento enlaces | `bgp/edges.csv`, `ripe_atlas/edges.csv` + nodos de ambas fuentes | `src_id`+`dst_id` en edges y `node_id`+`asn` en nodes (o `src_asn`+`dst_asn`) |
| 7 | Distribución de grado (CCDF y log-log) | nodos/edges de BGP, RIPE y Combinado | `degree_total` **o** `in_degree`+`out_degree` (fallback cálculo desde edges) |
| 8 | Top 10 ASNs chilenos por relevancia | `merged/nodes.csv` (+ opcional `merged/edges.csv`) | `asn` y métrica (`degree_total` o `path_occurrences` o `betweenness`); opcional `name`, `country` |
| 9 | Top 10 enlaces AS-AS por frecuencia | `merged/edges.csv` + `merged/nodes.csv` | Enlaces: `src_id`+`dst_id` (o `src_asn`+`dst_asn`) y frecuencia (`freq_combined` o `weight`) |
| 10 | Grafo simplificado filtrado Chile | `merged/nodes.csv` + `merged/edges.csv` (opcional bgp/ripe nodes) | `asn` y enlaces; para color por fuente: `seen_in_bgp`/`seen_in_ripe` (o se infiere); para color por PDB: `has_peeringdb` o atributos |

Si faltan columnas para un gráfico, el notebook imprime advertencia y continúa con el resto.


In [4]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
import networkx as nx

try:
    from pyvis.network import Network
    PYVIS_AVAILABLE = True
except Exception:
    PYVIS_AVAILABLE = False


# Configuración global de salida
DPI = 350
DEFAULT_OUTPUT_DIR = Path("output_figures")

SOURCE_COLORS = {
    "BGP": "#2E86AB",
    "RIPE": "#A23B72",
    "Combinado": "#F18F01",
}

COLUMN_ALIASES: Dict[str, List[str]] = {
    "asn": ["asn", "as_number", "autonomous_system", "autonomoussystem", "asn_id"],
    "node_id": ["node_id", "nodeid", "id", "node", "idx"],
    "name": ["name", "as_name", "asn_name", "org_name", "network_name"],
    "country": ["country", "country_code", "cc", "iso2", "countrycode", "nation"],
    "region": ["region", "state", "administrative_region", "admin_region", "provincia"],
    "city": ["city", "town", "ciudad", "locality"],
    "lat": ["lat", "latitude", "probe_lat", "latitud", "geo_lat"],
    "lon": ["lon", "lng", "longitude", "probe_lon", "longitud", "geo_lon"],
    "src_id": ["src_id", "src", "source", "from", "u", "node1_id"],
    "dst_id": ["dst_id", "dst", "target", "to", "v", "node2_id"],
    "src_asn": ["src_asn", "source_asn", "asn_src", "from_asn"],
    "dst_asn": ["dst_asn", "target_asn", "asn_dst", "to_asn"],
    "weight": ["weight", "count", "frequency", "freq", "times", "n"],
    "freq_bgp": ["freq_bgp", "bgp_freq", "frequency_bgp", "weight_bgp"],
    "freq_ripe": ["freq_ripe", "ripe_freq", "frequency_ripe", "weight_ripe"],
    "freq_combined": ["freq_combined", "combined_freq", "frequency_combined", "weight"],
    "in_degree": ["in_degree", "indegree", "deg_in", "in_deg"],
    "out_degree": ["out_degree", "outdegree", "deg_out", "out_deg"],
    "degree_total": ["degree_total", "total_degree", "degree", "deg_total", "k"],
    "path_occurrences": ["path_occurrences", "path_count", "paths", "occurrences"],
    "betweenness": ["betweenness", "betweenness_centrality", "bc"],
    "as_type": ["as_type", "type", "asn_type", "asn_class", "classification"],
    "has_peeringdb": ["has_peeringdb", "has_pdb", "peeringdb", "pdb_present"],
    "seen_in_bgp": ["seen_in_bgp", "in_bgp", "bgp_seen", "source_bgp"],
    "seen_in_ripe": ["seen_in_ripe", "in_ripe", "ripe_seen", "source_ripe"],
    "category": ["category", "categoria", "metric", "métrica", "tipo"],
    "value": ["value", "valor", "count", "cantidad", "total", "n"],
}


def _norm(text: object) -> str:
    """Normaliza textos para matching de aliases (case-insensitive y sin símbolos)."""
    return "".join(ch for ch in str(text).lower() if ch.isalnum())


def _warn(msg: str) -> None:
    """Imprime advertencias sin interrumpir ejecución."""
    print(f"[ADVERTENCIA] {msg}")


def find_any_column(df: pd.DataFrame, aliases: Iterable[str]) -> Optional[str]:
    """Retorna la primera columna existente en df que coincide con aliases."""
    norm_map = {_norm(col): col for col in df.columns}
    for alias in aliases:
        key = _norm(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def get_column(df: pd.DataFrame, logical_name: str, extra_aliases: Optional[Iterable[str]] = None) -> Optional[str]:
    """Busca una columna por nombre lógico + aliases opcionales."""
    aliases = list(COLUMN_ALIASES.get(logical_name, []))
    if extra_aliases:
        aliases.extend(list(extra_aliases))
    return find_any_column(df, aliases)


def choose_existing_path(candidates: Iterable[Path]) -> Optional[Path]:
    """Retorna el primer archivo existente entre varios candidatos."""
    for path in candidates:
        if path.exists():
            return path
    return None


def read_csv_safe(path: Optional[Path], label: str) -> Optional[pd.DataFrame]:
    """Lee CSV de forma segura y retorna None si falla."""
    if path is None:
        _warn(f"No se definió ruta para {label}.")
        return None

    if not path.exists():
        _warn(f"No existe {label}: {path}")
        return None

    try:
        return pd.read_csv(path)
    except Exception as exc:
        _warn(f"Error al leer {label} ({path}): {exc}")
        return None


def series_nonempty(series: pd.Series) -> pd.Series:
    """Detecta valores no vacíos en series numéricas, booleanas o texto."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce").fillna(0)
        return numeric != 0

    text = series.fillna("").astype(str).str.strip().str.lower()
    return ~text.isin(["", "nan", "none", "null", "na"])


def save_figure(fig: plt.Figure, output_dir: Path, filename: str) -> None:
    """Guarda figura en PNG de alta resolución."""
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / filename
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Figura guardada: {out_path}")


def build_has_attributes_mask(nodes_df: pd.DataFrame) -> Optional[pd.Series]:
    """Construye máscara booleana de nodos con atributos PeeringDB."""
    if nodes_df is None or nodes_df.empty:
        return None

    has_col = get_column(nodes_df, "has_peeringdb")
    if has_col:
        return series_nonempty(nodes_df[has_col])

    attr_cols = []
    for col in nodes_df.columns:
        low = str(col).lower()
        if low.startswith("info_") or low.startswith("policy_"):
            attr_cols.append(col)
        if low in {"ix_count", "fac_count", "org_id"}:
            attr_cols.append(col)
        if "peeringdb" in low:
            attr_cols.append(col)

    attr_cols = list(dict.fromkeys(attr_cols))

    if not attr_cols:
        _warn("No se encontraron columnas para inferir atributos PeeringDB.")
        return None

    mask = pd.Series(False, index=nodes_df.index)
    for col in attr_cols:
        mask = mask | series_nonempty(nodes_df[col])

    return mask


def extract_degree_series(nodes_df: pd.DataFrame, edges_df: Optional[pd.DataFrame] = None) -> Optional[pd.Series]:
    """Obtiene degree_total desde columna directa, in+out o cálculo por aristas."""
    if nodes_df is None or nodes_df.empty:
        return None

    degree_col = get_column(nodes_df, "degree_total")
    if degree_col:
        return pd.to_numeric(nodes_df[degree_col], errors="coerce").fillna(0)

    in_col = get_column(nodes_df, "in_degree")
    out_col = get_column(nodes_df, "out_degree")
    if in_col and out_col:
        in_deg = pd.to_numeric(nodes_df[in_col], errors="coerce").fillna(0)
        out_deg = pd.to_numeric(nodes_df[out_col], errors="coerce").fillna(0)
        return in_deg + out_deg

    if edges_df is not None:
        node_col = get_column(nodes_df, "node_id")
        src_col = get_column(edges_df, "src_id")
        dst_col = get_column(edges_df, "dst_id")

        if node_col and src_col and dst_col:
            all_ends = pd.concat([edges_df[src_col], edges_df[dst_col]], ignore_index=True)
            counts = all_ends.value_counts()
            return nodes_df[node_col].map(counts).fillna(0)

    _warn("No fue posible calcular degree_total (faltan columnas).")
    return None


def filter_chile_rows(df: pd.DataFrame) -> Tuple[pd.DataFrame, bool]:
    """Filtra filas de Chile si existe columna país; retorna df y bandera de filtro usado."""
    country_col = get_column(df, "country")
    if not country_col:
        return df, False

    values = df[country_col].fillna("").astype(str).str.strip().str.upper()
    mask = values.isin({"CL", "CHL"}) | values.str.contains("CHILE", na=False)

    if mask.any():
        return df[mask].copy(), True

    _warn("Existe columna de país, pero no se detectaron filas de Chile. Se usa dataset completo.")
    return df, True


def extract_asn_set(nodes_df: pd.DataFrame) -> Optional[set]:
    """Extrae conjunto de ASNs desde nodes_df."""
    if nodes_df is None or nodes_df.empty:
        return None

    asn_col = get_column(nodes_df, "asn")
    if not asn_col:
        _warn("Falta columna ASN en nodos.")
        return None

    asn_values = pd.to_numeric(nodes_df[asn_col], errors="coerce").dropna().astype(int)
    return set(asn_values.tolist())


def extract_asn_edge_table(edges_df: pd.DataFrame, nodes_df: pd.DataFrame) -> Optional[pd.DataFrame]:
    """Convierte aristas a formato ASN-ASN usando src/dst ASN directo o mapeo node_id -> asn."""
    if edges_df is None or nodes_df is None or edges_df.empty or nodes_df.empty:
        return None

    src_asn_col = get_column(edges_df, "src_asn")
    dst_asn_col = get_column(edges_df, "dst_asn")

    edge_table = pd.DataFrame(index=edges_df.index)

    if src_asn_col and dst_asn_col:
        edge_table["asn_u"] = pd.to_numeric(edges_df[src_asn_col], errors="coerce")
        edge_table["asn_v"] = pd.to_numeric(edges_df[dst_asn_col], errors="coerce")
    else:
        src_col = get_column(edges_df, "src_id")
        dst_col = get_column(edges_df, "dst_id")
        node_col = get_column(nodes_df, "node_id")
        asn_col = get_column(nodes_df, "asn")

        if not (src_col and dst_col and node_col and asn_col):
            _warn("No se pudo mapear aristas a ASN-ASN (faltan src/dst o node_id/asn).")
            return None

        mapping = dict(zip(nodes_df[node_col], nodes_df[asn_col]))
        edge_table["asn_u"] = edges_df[src_col].map(mapping)
        edge_table["asn_v"] = edges_df[dst_col].map(mapping)

    edge_table = edge_table.dropna().copy()
    if edge_table.empty:
        return None

    edge_table["asn_u"] = edge_table["asn_u"].astype(int)
    edge_table["asn_v"] = edge_table["asn_v"].astype(int)

    edge_table[["asn_u", "asn_v"]] = np.sort(edge_table[["asn_u", "asn_v"]].values, axis=1)
    return edge_table


def extract_asn_edge_set(edges_df: pd.DataFrame, nodes_df: pd.DataFrame) -> Optional[set]:
    """Devuelve set de tuplas ASN-ASN sin dirección."""
    table = extract_asn_edge_table(edges_df, nodes_df)
    if table is None or table.empty:
        return None
    return set(zip(table["asn_u"], table["asn_v"]))


def get_metric_series(nodes_df: pd.DataFrame, edges_df: Optional[pd.DataFrame], metric: str) -> Tuple[Optional[pd.Series], str]:
    """Devuelve serie de métrica solicitada y nombre efectivo usado."""
    metric = (metric or "degree_total").strip().lower()

    if metric == "degree_total":
        deg = extract_degree_series(nodes_df, edges_df)
        return deg, "degree_total"

    col = get_column(nodes_df, metric)
    if col:
        values = pd.to_numeric(nodes_df[col], errors="coerce").fillna(0)
        return values, metric

    _warn(f"No existe métrica '{metric}'. Se usa 'degree_total'.")
    deg = extract_degree_series(nodes_df, edges_df)
    return deg, "degree_total"


def plot_ripe_targets_bars(ripe_targets_df: Optional[pd.DataFrame], output_dir: Path) -> None:
    if ripe_targets_df is None or ripe_targets_df.empty:
        _warn("Gráfico 1 omitido: no hay CSV de targets RIPE Atlas.")
        return

    expected = {
        "Prefijos Chile": ["prefijos_chile", "chile_prefixes", "prefijoscl", "prefijos"],
        "Targets generados": ["targets_generados", "generated_targets", "targets"],
        "Con respuesta": ["con_respuesta", "with_response", "responsive_targets", "success"],
        "Sin respuesta": ["sin_respuesta", "without_response", "failed_targets", "no_response"],
    }

    values: Dict[str, float] = {}
    for label, aliases in expected.items():
        col = find_any_column(ripe_targets_df, aliases)
        if col:
            values[label] = float(pd.to_numeric(ripe_targets_df[col], errors="coerce").fillna(0).sum())

    if not values:
        _warn("Gráfico 1 omitido: no se detectaron columnas de métricas de targets.")
        return

    labels = ["Prefijos Chile", "Targets generados", "Con respuesta", "Sin respuesta"]
    y = [int(round(values.get(label, 0))) for label in labels]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, y, color=["#2E86AB", "#F18F01", "#4CAF50", "#C73E1D"] )
    for bar, val in zip(bars, y):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.set_title("RIPE Atlas: Resumen de Targets para Chile", fontsize=13, fontweight="bold")
    ax.set_ylabel("Cantidad")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", alpha=0.2)
    save_figure(fig, output_dir, "01_ripe_targets_barras.png")


def plot_probe_geographic_distribution(ripe_probes_df: Optional[pd.DataFrame], output_dir: Path) -> None:
    if ripe_probes_df is None or ripe_probes_df.empty:
        _warn("Gráfico 2 omitido: no hay CSV de probes RIPE Atlas.")
        return

    lat_col = get_column(ripe_probes_df, "lat")
    lon_col = get_column(ripe_probes_df, "lon")
    region_col = get_column(ripe_probes_df, "region")

    if lat_col and lon_col:
        lat = pd.to_numeric(ripe_probes_df[lat_col], errors="coerce")
        lon = pd.to_numeric(ripe_probes_df[lon_col], errors="coerce")
        valid = lat.notna() & lon.notna()

        if valid.any():
            fig, ax = plt.subplots(figsize=(8, 10))
            ax.scatter(lon[valid], lat[valid], s=25, alpha=0.75, color="#2E86AB", edgecolor="white", linewidth=0.3)
            ax.set_title("Probes chilenos RIPE Atlas (lat/lon)", fontsize=13, fontweight="bold")
            ax.set_xlabel("Longitud")
            ax.set_ylabel("Latitud")
            ax.grid(alpha=0.2)
            save_figure(fig, output_dir, "02_probes_distribucion_geografica.png")
            return

    if region_col:
        counts = ripe_probes_df[region_col].fillna("Desconocido").astype(str).value_counts().head(20)
        fig, ax = plt.subplots(figsize=(11, 6))
        bars = ax.bar(counts.index.astype(str), counts.values, color="#A23B72")
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(val)}", ha="center", va="bottom", fontsize=9)
        ax.set_title("Probes RIPE Atlas por región (Chile)", fontsize=13, fontweight="bold")
        ax.set_ylabel("Cantidad de probes")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.2)
        save_figure(fig, output_dir, "02_probes_por_region.png")
        return

    _warn("Gráfico 2 omitido: no hay lat/lon ni región para probes.")


def plot_peeringdb_coverage(nodes_df: Optional[pd.DataFrame], output_dir: Path) -> None:
    if nodes_df is None or nodes_df.empty:
        _warn("Gráfico 3 omitido: no hay nodos para cobertura PeeringDB.")
        return

    has_attr = build_has_attributes_mask(nodes_df)
    if has_attr is None:
        _warn("Gráfico 3 omitido: no se pudo inferir cobertura de atributos.")
        return

    total = len(nodes_df)
    con_attr = int(has_attr.sum())
    sin_attr = int(total - con_attr)
    coverage_pct = (con_attr / total * 100) if total > 0 else 0

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(["Con atributos", "Sin atributos"], [con_attr, sin_attr], color=["#4CAF50", "#C73E1D"])
    for bar, val in zip(bars, [con_attr, sin_attr]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(f"Cobertura de atributos PeeringDB en topología Chile ({coverage_pct:.1f}%)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Cantidad de ASNs")
    ax.grid(axis="y", alpha=0.2)
    save_figure(fig, output_dir, "03_peeringdb_cobertura.png")


def plot_asn_types_without_attributes(nodes_df: Optional[pd.DataFrame], edges_df: Optional[pd.DataFrame], output_dir: Path) -> None:
    if nodes_df is None or nodes_df.empty:
        _warn("Gráfico 4 omitido: no hay nodos.")
        return

    has_attr = build_has_attributes_mask(nodes_df)
    if has_attr is None:
        _warn("Gráfico 4 omitido: no se pudo identificar ASNs sin atributos.")
        return

    subset = nodes_df.loc[~has_attr].copy()
    if subset.empty:
        _warn("Gráfico 4 omitido: no hay ASNs sin atributos.")
        return

    as_type_col = get_column(subset, "as_type")
    if as_type_col:
        normalized = subset[as_type_col].fillna("desconocido").astype(str).str.strip().str.lower()
        normalized = normalized.replace({"": "desconocido"})
        counts = normalized.value_counts()

        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.bar(counts.index.astype(str), counts.values, color="#F18F01")
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(val)}", ha="center", va="bottom", fontsize=9)
        ax.set_title("ASNs sin atributos PeeringDB por tipo", fontsize=13, fontweight="bold")
        ax.set_ylabel("Cantidad")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.2)
        save_figure(fig, output_dir, "04_asns_sin_atributos_por_tipo.png")
        return

    degree = extract_degree_series(subset, edges_df)
    if degree is None:
        _warn("Gráfico 4 omitido: no existe as_type ni degree_total para bucketización.")
        return

    q1, q2 = degree.quantile([1/3, 2/3]).tolist()
    labels = pd.cut(degree, bins=[-np.inf, q1, q2, np.inf], labels=["bajo", "medio", "alto"], include_lowest=True)
    counts = labels.value_counts().reindex(["bajo", "medio", "alto"]).fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(counts.index.astype(str), counts.values, color=["#8ECAE6", "#FFB703", "#FB8500"])
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(val)}", ha="center", va="bottom", fontsize=10)
    ax.set_title("ASNs sin atributos PeeringDB por bucket de grado", fontsize=13, fontweight="bold")
    ax.set_ylabel("Cantidad")
    ax.grid(axis="y", alpha=0.2)
    save_figure(fig, output_dir, "04_asns_sin_atributos_por_bucket_grado.png")

def plot_topology_size_comparison(
    bgp_nodes: Optional[pd.DataFrame],
    bgp_edges: Optional[pd.DataFrame],
    ripe_nodes: Optional[pd.DataFrame],
    ripe_edges: Optional[pd.DataFrame],
    merged_nodes: Optional[pd.DataFrame],
    merged_edges: Optional[pd.DataFrame],
    output_dir: Path,
) -> None:
    """5) Comparación de tamaño entre topologías."""
    node_counts = {
        "BGP": int(len(bgp_nodes)) if bgp_nodes is not None else 0,
        "RIPE": int(len(ripe_nodes)) if ripe_nodes is not None else 0,
        "Combinado": int(len(merged_nodes)) if merged_nodes is not None else 0,
    }
    edge_counts = {
        "BGP": int(len(bgp_edges)) if bgp_edges is not None else 0,
        "RIPE": int(len(ripe_edges)) if ripe_edges is not None else 0,
        "Combinado": int(len(merged_edges)) if merged_edges is not None else 0,
    }

    fig1, ax1 = plt.subplots(figsize=(8, 5))
    bars1 = ax1.bar(list(node_counts.keys()), list(node_counts.values()), color=[SOURCE_COLORS["BGP"], SOURCE_COLORS["RIPE"], SOURCE_COLORS["Combinado"]])
    for b, v in zip(bars1, node_counts.values()):
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}", ha="center", va="bottom", fontsize=10)
    ax1.set_title("Nodos por fuente (BGP, RIPE, Combinado)", fontsize=13, fontweight="bold")
    ax1.set_ylabel("Cantidad de nodos")
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax1.grid(axis="y", alpha=0.2)
    save_figure(fig1, output_dir, "05a_nodos_por_fuente.png")

    fig2, ax2 = plt.subplots(figsize=(8, 5))
    bars2 = ax2.bar(list(edge_counts.keys()), list(edge_counts.values()), color=[SOURCE_COLORS["BGP"], SOURCE_COLORS["RIPE"], SOURCE_COLORS["Combinado"]])
    for b, v in zip(bars2, edge_counts.values()):
        ax2.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}", ha="center", va="bottom", fontsize=10)
    ax2.set_title("Aristas por fuente (BGP, RIPE, Combinado)", fontsize=13, fontweight="bold")
    ax2.set_ylabel("Cantidad de aristas")
    ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.grid(axis="y", alpha=0.2)
    save_figure(fig2, output_dir, "05b_aristas_por_fuente.png")


def _plot_single_stacked_bar(values: Dict[str, int], title: str, x_label: str, output_dir: Path, filename: str) -> None:
    """Helper para crear barra apilada de 3 categorías."""
    labels = list(values.keys())
    numbers = list(values.values())
    colors = ["#6A8EAE", "#F18F01", "#C17C9F"]

    fig, ax = plt.subplots(figsize=(7, 5))
    bottom = 0
    for i, (label, val) in enumerate(zip(labels, numbers)):
        ax.bar([x_label], [val], bottom=[bottom], label=label, color=colors[i])
        if val > 0:
            ax.text(0, bottom + val / 2, f"{val}", ha="center", va="center", color="white", fontweight="bold")
        bottom += val

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("Cantidad")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    save_figure(fig, output_dir, filename)


def plot_source_overlap(
    bgp_nodes: Optional[pd.DataFrame],
    ripe_nodes: Optional[pd.DataFrame],
    bgp_edges: Optional[pd.DataFrame],
    ripe_edges: Optional[pd.DataFrame],
    output_dir: Path,
) -> None:
    """6) Solapamiento entre fuentes para ASNs y enlaces."""
    asn_bgp = extract_asn_set(bgp_nodes) if bgp_nodes is not None else None
    asn_ripe = extract_asn_set(ripe_nodes) if ripe_nodes is not None else None

    if asn_bgp is None or asn_ripe is None:
        _warn("Gráfico 6a omitido: faltan ASNs en nodos de BGP o RIPE.")
    else:
        only_bgp = asn_bgp - asn_ripe
        common = asn_bgp & asn_ripe
        only_ripe = asn_ripe - asn_bgp
        _plot_single_stacked_bar(
            {
                "Solo BGP": len(only_bgp),
                "Comunes": len(common),
                "Solo RIPE": len(only_ripe),
            },
            "Solapamiento de ASNs entre fuentes",
            "ASNs",
            output_dir,
            "06a_solapamiento_asns.png",
        )

    if any(v is None for v in [bgp_nodes, ripe_nodes, bgp_edges, ripe_edges]):
        _warn("Gráfico 6b omitido: faltan nodos/aristas de BGP o RIPE.")
        return

    edge_bgp = extract_asn_edge_set(bgp_edges, bgp_nodes)
    edge_ripe = extract_asn_edge_set(ripe_edges, ripe_nodes)
    if edge_bgp is None or edge_ripe is None:
        _warn("Gráfico 6b omitido: no se pudieron mapear enlaces a ASN-ASN.")
        return

    only_bgp = edge_bgp - edge_ripe
    common = edge_bgp & edge_ripe
    only_ripe = edge_ripe - edge_bgp
    _plot_single_stacked_bar(
        {
            "Solo BGP": len(only_bgp),
            "Comunes": len(common),
            "Solo RIPE": len(only_ripe),
        },
        "Solapamiento de enlaces entre fuentes",
        "Enlaces",
        output_dir,
        "06b_solapamiento_enlaces.png",
    )


def compute_ccdf(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Calcula CCDF para un arreglo de grados."""
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if values.size == 0:
        return np.array([]), np.array([])

    x = np.sort(np.unique(values))
    y = np.array([(values >= xi).mean() for xi in x])
    return x, y


def plot_degree_distribution(
    bgp_nodes: Optional[pd.DataFrame],
    bgp_edges: Optional[pd.DataFrame],
    ripe_nodes: Optional[pd.DataFrame],
    ripe_edges: Optional[pd.DataFrame],
    merged_nodes: Optional[pd.DataFrame],
    merged_edges: Optional[pd.DataFrame],
    output_dir: Path,
) -> None:
    """7) Distribución de grado (CCDF lineal y log-log)."""
    sources = {
        "BGP": (bgp_nodes, bgp_edges, SOURCE_COLORS["BGP"]),
        "RIPE": (ripe_nodes, ripe_edges, SOURCE_COLORS["RIPE"]),
        "Combinado": (merged_nodes, merged_edges, SOURCE_COLORS["Combinado"]),
    }

    curves = {}
    for name, (nodes_df, edges_df, color) in sources.items():
        if nodes_df is None:
            continue
        degree = extract_degree_series(nodes_df, edges_df)
        if degree is None:
            continue
        x, y = compute_ccdf(pd.to_numeric(degree, errors="coerce").fillna(0).to_numpy(dtype=float))
        if x.size > 0:
            curves[name] = (x, y, color)

    if not curves:
        _warn("Gráfico 7 omitido: no se pudieron calcular grados para BGP/RIPE/Combinado.")
        return

    fig1, ax1 = plt.subplots(figsize=(9, 6))
    for name, (x, y, color) in curves.items():
        ax1.step(x, y, where="post", label=name, color=color, linewidth=2)
    ax1.set_title("Distribución de grado (CCDF)", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Grado")
    ax1.set_ylabel("P(K ≥ k)")
    ax1.grid(alpha=0.2)
    ax1.legend()
    save_figure(fig1, output_dir, "07a_distribucion_grado_ccdf.png")

    fig2, ax2 = plt.subplots(figsize=(9, 6))
    for name, (x, y, color) in curves.items():
        mask = (x > 0) & (y > 0)
        if mask.any():
            ax2.step(x[mask], y[mask], where="post", label=name, color=color, linewidth=2)
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    ax2.set_title("Distribución de grado (CCDF log-log)", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Grado (log)")
    ax2.set_ylabel("P(K ≥ k) (log)")
    ax2.grid(alpha=0.2, which="both")
    ax2.legend()
    save_figure(fig2, output_dir, "07b_distribucion_grado_ccdf_loglog.png")


def plot_top10_chilean_asns(
    nodes_df: Optional[pd.DataFrame],
    edges_df: Optional[pd.DataFrame],
    output_dir: Path,
    metric: str = "degree_total",
) -> None:
    """8) Top 10 ASNs chilenos por relevancia."""
    if nodes_df is None or nodes_df.empty:
        _warn("Gráfico 8 omitido: no hay nodos disponibles.")
        return

    df, used_country_filter = filter_chile_rows(nodes_df.copy())
    if df.empty:
        _warn("Gráfico 8 omitido: el filtro de Chile dejó 0 filas.")
        return

    asn_col = get_column(df, "asn")
    if not asn_col:
        _warn("Gráfico 8 omitido: falta columna ASN.")
        return

    metric_values, metric_used = get_metric_series(df, edges_df, metric)
    if metric_values is None:
        _warn("Gráfico 8 omitido: no se pudo calcular la métrica seleccionada.")
        return

    df["_metric"] = pd.to_numeric(metric_values, errors="coerce").fillna(0)
    name_col = get_column(df, "name")

    top = df.nlargest(10, "_metric").copy()
    if top.empty:
        _warn("Gráfico 8 omitido: ranking vacío.")
        return

    labels = []
    for _, row in top.iterrows():
        asn_val = int(pd.to_numeric(row[asn_col], errors="coerce"))
        if name_col and str(row[name_col]).strip() and str(row[name_col]).strip().lower() != "nan":
            labels.append(f"AS{asn_val} - {row[name_col]}")
        else:
            labels.append(f"AS{asn_val}")

    values = top["_metric"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(labels[::-1], values[::-1], color="#1982C4")
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height() / 2, f" {width:.2f}", va="center", fontsize=8)

    suffix = "(Chile)" if used_country_filter else "(sin filtro país)"
    ax.set_title(f"Top 10 ASNs por {metric_used} {suffix}", fontsize=13, fontweight="bold")
    ax.set_xlabel(metric_used)
    ax.grid(axis="x", alpha=0.2)
    save_figure(fig, output_dir, f"08_top10_asns_por_{metric_used}.png")


def plot_top10_links_frequency(
    edges_df: Optional[pd.DataFrame],
    nodes_df: Optional[pd.DataFrame],
    output_dir: Path,
) -> None:
    """9) Top 10 enlaces AS-AS por frecuencia combinada."""
    if edges_df is None or nodes_df is None or edges_df.empty or nodes_df.empty:
        _warn("Gráfico 9 omitido: faltan aristas/nodos combinados.")
        return

    edge_pairs = extract_asn_edge_table(edges_df, nodes_df)
    if edge_pairs is None or edge_pairs.empty:
        _warn("Gráfico 9 omitido: no se pudieron mapear enlaces ASN-ASN.")
        return

    idx = edge_pairs.index
    work = edge_pairs.copy()

    freq_combined_col = get_column(edges_df, "freq_combined") or get_column(edges_df, "weight")
    if freq_combined_col:
        work["freq_combined"] = pd.to_numeric(edges_df.loc[idx, freq_combined_col], errors="coerce").fillna(0).to_numpy()
    else:
        work["freq_combined"] = 1.0

    freq_bgp_col = get_column(edges_df, "freq_bgp")
    if freq_bgp_col:
        work["freq_bgp"] = pd.to_numeric(edges_df.loc[idx, freq_bgp_col], errors="coerce").fillna(0).to_numpy()

    freq_ripe_col = get_column(edges_df, "freq_ripe")
    if freq_ripe_col:
        work["freq_ripe"] = pd.to_numeric(edges_df.loc[idx, freq_ripe_col], errors="coerce").fillna(0).to_numpy()

    agg = {"freq_combined": "sum"}
    if "freq_bgp" in work.columns:
        agg["freq_bgp"] = "sum"
    if "freq_ripe" in work.columns:
        agg["freq_ripe"] = "sum"

    ranked = (
        work.groupby(["asn_u", "asn_v"], as_index=False)
        .agg(agg)
        .sort_values("freq_combined", ascending=False)
        .head(10)
    )

    if ranked.empty:
        _warn("Gráfico 9 omitido: ranking de enlaces vacío.")
        return

    ranked["label"] = ranked.apply(lambda r: f"AS{int(r['asn_u'])}-AS{int(r['asn_v'])}", axis=1)

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(ranked["label"][::-1], ranked["freq_combined"][::-1], color="#8AC926")

    for i, (_, row) in enumerate(ranked.iloc[::-1].iterrows()):
        extras = []
        if "freq_bgp" in row.index:
            extras.append(f"BGP={row['freq_bgp']:.0f}")
        if "freq_ripe" in row.index:
            extras.append(f"RIPE={row['freq_ripe']:.0f}")
        suffix = f" ({', '.join(extras)})" if extras else ""
        ax.text(row["freq_combined"], i, f" {row['freq_combined']:.0f}{suffix}", va="center", fontsize=8)

    ax.set_title("Top 10 enlaces AS-AS por frecuencia", fontsize=13, fontweight="bold")
    ax.set_xlabel("Frecuencia combinada")
    ax.grid(axis="x", alpha=0.2)
    save_figure(fig, output_dir, "09_top10_enlaces_frecuencia.png")


def _build_source_presence_maps(
    merged_nodes: pd.DataFrame,
    bgp_nodes: Optional[pd.DataFrame],
    ripe_nodes: Optional[pd.DataFrame],
) -> Tuple[Dict[int, bool], Dict[int, bool]]:
    """Construye mapas ASN->seen_in_bgp/seen_in_ripe usando columnas o inferencia por sets."""
    asn_col = get_column(merged_nodes, "asn")
    if not asn_col:
        return {}, {}

    asn_values = pd.to_numeric(merged_nodes[asn_col], errors="coerce")
    valid = asn_values.notna()

    seen_bgp_col = get_column(merged_nodes, "seen_in_bgp")
    seen_ripe_col = get_column(merged_nodes, "seen_in_ripe")

    if seen_bgp_col and seen_ripe_col:
        bgp_map = dict(zip(asn_values[valid].astype(int), series_nonempty(merged_nodes.loc[valid, seen_bgp_col]).astype(bool)))
        ripe_map = dict(zip(asn_values[valid].astype(int), series_nonempty(merged_nodes.loc[valid, seen_ripe_col]).astype(bool)))
        return bgp_map, ripe_map

    bgp_set = extract_asn_set(bgp_nodes) if bgp_nodes is not None else set()
    ripe_set = extract_asn_set(ripe_nodes) if ripe_nodes is not None else set()

    bgp_map = {}
    ripe_map = {}
    for asn in asn_values[valid].astype(int):
        bgp_map[asn] = asn in bgp_set
        ripe_map[asn] = asn in ripe_set
    return bgp_map, ripe_map


def export_pyvis_graph(graph: nx.Graph, node_colors: Dict[int, str], node_sizes: Dict[int, float], output_html: Path) -> None:
    """Exporta HTML interactivo con PyVis (opcional)."""
    if not PYVIS_AVAILABLE:
        _warn("PyVis no está instalado. Se omite exportación interactiva HTML.")
        return

    net = Network(height="760px", width="100%", bgcolor="#ffffff", font_color="#1f2937")
    net.barnes_hut(gravity=-18000, central_gravity=0.2, spring_length=150, spring_strength=0.02, damping=0.09)

    for n in graph.nodes():
        net.add_node(
            int(n),
            label=f"AS{int(n)}",
            title=f"AS{int(n)}",
            color=node_colors.get(int(n), "#9AA0A6"),
            size=max(8, float(node_sizes.get(int(n), 120)) / 25.0),
        )

    for u, v in graph.edges():
        net.add_edge(int(u), int(v))

    output_html.parent.mkdir(parents=True, exist_ok=True)
    net.write_html(str(output_html), open_browser=False, notebook=False)
    print(f"[OK] Visualización interactiva guardada: {output_html}")


def plot_simplified_chile_graph(
    merged_nodes: Optional[pd.DataFrame],
    merged_edges: Optional[pd.DataFrame],
    bgp_nodes: Optional[pd.DataFrame],
    ripe_nodes: Optional[pd.DataFrame],
    output_dir: Path,
    top_n: int = 80,
    color_mode: str = "source",
    export_pyvis_html: bool = True,
) -> None:
    """10) Visualización simplificada del grafo filtrado de Chile."""
    if merged_nodes is None or merged_edges is None:
        _warn("Gráfico 10 omitido: faltan CSV combinados de nodos/aristas.")
        return

    asn_col = get_column(merged_nodes, "asn")
    if not asn_col:
        _warn("Gráfico 10 omitido: falta columna ASN en merged nodes.")
        return

    nodes_cl, filtered = filter_chile_rows(merged_nodes.copy())
    if nodes_cl.empty:
        _warn("Filtro de Chile devolvió 0 filas; se usa dataset combinado completo.")
        nodes_cl = merged_nodes.copy()

    asn_subset = set(pd.to_numeric(nodes_cl[asn_col], errors="coerce").dropna().astype(int).tolist())
    edge_pairs = extract_asn_edge_table(merged_edges, merged_nodes)
    if edge_pairs is None or edge_pairs.empty:
        _warn("Gráfico 10 omitido: no se pudieron construir enlaces ASN-ASN.")
        return

    edge_pairs = edge_pairs[edge_pairs["asn_u"].isin(asn_subset) & edge_pairs["asn_v"].isin(asn_subset)]
    if edge_pairs.empty:
        _warn("Gráfico 10 omitido: no hay enlaces para el subconjunto de Chile.")
        return

    G = nx.Graph()
    G.add_edges_from(edge_pairs[["asn_u", "asn_v"]].itertuples(index=False, name=None))
    G.add_nodes_from(asn_subset)

    if G.number_of_nodes() == 0:
        _warn("Gráfico 10 omitido: grafo vacío.")
        return

    degree_full = dict(G.degree())
    top_nodes = sorted(degree_full, key=degree_full.get, reverse=True)[:max(1, int(top_n))]
    H = G.subgraph(top_nodes).copy()

    if H.number_of_nodes() == 0:
        _warn("Gráfico 10 omitido: subgrafo top-N vacío.")
        return

    has_attr = build_has_attributes_mask(merged_nodes)
    has_attr_map: Dict[int, bool] = {}
    if has_attr is not None:
        asn_vals = pd.to_numeric(merged_nodes[asn_col], errors="coerce")
        valid = asn_vals.notna()
        has_attr_map = dict(zip(asn_vals[valid].astype(int), has_attr[valid].astype(bool)))

    seen_bgp_map, seen_ripe_map = _build_source_presence_maps(merged_nodes, bgp_nodes, ripe_nodes)

    node_sizes = {n: 120 + 45 * H.degree(n) for n in H.nodes()}
    node_colors: Dict[int, str] = {}

    if str(color_mode).lower() == "has_peeringdb":
        for n in H.nodes():
            node_colors[n] = "#2A9D8F" if has_attr_map.get(int(n), False) else "#E76F51"
        legend = [
            Patch(facecolor="#2A9D8F", edgecolor="black", label="Con atributos PeeringDB"),
            Patch(facecolor="#E76F51", edgecolor="black", label="Sin atributos PeeringDB"),
        ]
    else:
        for n in H.nodes():
            in_bgp = seen_bgp_map.get(int(n), False)
            in_ripe = seen_ripe_map.get(int(n), False)
            if in_bgp and in_ripe:
                node_colors[n] = "#6A4C93"
            elif in_bgp:
                node_colors[n] = "#1982C4"
            elif in_ripe:
                node_colors[n] = "#FF595E"
            else:
                node_colors[n] = "#ADB5BD"
        legend = [
            Patch(facecolor="#1982C4", edgecolor="black", label="Solo BGP"),
            Patch(facecolor="#6A4C93", edgecolor="black", label="BGP y RIPE"),
            Patch(facecolor="#FF595E", edgecolor="black", label="Solo RIPE"),
            Patch(facecolor="#ADB5BD", edgecolor="black", label="Sin marca de fuente"),
        ]

    fig, ax = plt.subplots(figsize=(14, 10))
    k_val = 1.8 / np.sqrt(max(H.number_of_nodes(), 1))
    pos = nx.spring_layout(H, seed=42, k=k_val, iterations=250)

    nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.25, width=0.8, edge_color="#6c757d")
    nx.draw_networkx_nodes(
        H,
        pos,
        ax=ax,
        node_size=[node_sizes[n] for n in H.nodes()],
        node_color=[node_colors[n] for n in H.nodes()],
        linewidths=0.5,
        edgecolors="black",
        alpha=0.95,
    )

    if H.number_of_nodes() <= 45:
        labels = {n: f"AS{int(n)}" for n in H.nodes()}
    else:
        top_label_nodes = sorted(H.nodes(), key=lambda x: H.degree(x), reverse=True)[:25]
        labels = {n: f"AS{int(n)}" for n in top_label_nodes}
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=8, ax=ax)

    suffix = "(filtrado Chile)" if filtered else "(sin filtro de país)"
    ax.set_title(f"Grafo simplificado de topología AS {suffix} - Top {len(H.nodes())} nodos", fontsize=13, fontweight="bold")
    ax.axis("off")
    ax.legend(handles=legend, loc="upper right", frameon=True)
    save_figure(fig, output_dir, "10_grafo_chile_simplificado.png")

    if export_pyvis_html:
        export_pyvis_graph(H, node_colors=node_colors, node_sizes=node_sizes, output_html=output_dir / "10_grafo_chile_simplificado.html")


def main(
    base_data_dir: str = "data/csv",
    output_dir: str = "output_figures",
    ripe_targets_csv: Optional[str] = None,
    ripe_probes_csv: Optional[str] = None,
    top_asn_metric: str = "degree_total",
    top_n_graph: int = 80,
    graph_color_mode: str = "source",
) -> None:
    """Ejecuta los 10 gráficos solicitados y guarda PNG en output_figures/."""
    warnings.filterwarnings("ignore", category=UserWarning)

    base = Path(base_data_dir)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # Carga datasets principales por fuente
    bgp_nodes = read_csv_safe(base / "bgp/nodes.csv", "BGP nodes")
    bgp_edges = read_csv_safe(base / "bgp/edges.csv", "BGP edges")
    ripe_nodes = read_csv_safe(base / "ripe_atlas/nodes.csv", "RIPE nodes")
    ripe_edges = read_csv_safe(base / "ripe_atlas/edges.csv", "RIPE edges")
    merged_nodes = read_csv_safe(base / "merged/nodes.csv", "Merged nodes")
    merged_edges = read_csv_safe(base / "merged/edges.csv", "Merged edges")

    # Carga opcional para gráficos de targets/probes
    targets_path = Path(ripe_targets_csv) if ripe_targets_csv else choose_existing_path([
        base / "ripe_atlas/targets_summary.csv",
        base / "ripe_atlas/ripe_targets_summary.csv",
        base / "ripe_atlas/targets.csv",
        Path("data/ripe_targets_summary.csv"),
    ])
    probes_path = Path(ripe_probes_csv) if ripe_probes_csv else choose_existing_path([
        base / "ripe_atlas/probes.csv",
        base / "ripe_atlas/ripe_probes.csv",
        Path("data/ripe_probes.csv"),
    ])

    ripe_targets_df = read_csv_safe(targets_path, "RIPE targets") if targets_path else None
    ripe_probes_df = read_csv_safe(probes_path, "RIPE probes") if probes_path else None

    # 1) Barras de targets RIPE Atlas
    plot_ripe_targets_bars(ripe_targets_df, out)

    # 2) Distribución geográfica de probes chilenos
    plot_probe_geographic_distribution(ripe_probes_df, out)

    # 3) Cobertura de atributos PeeringDB
    plot_peeringdb_coverage(merged_nodes if merged_nodes is not None else bgp_nodes, out)

    # 4) Tipos de ASNs sin atributos
    plot_asn_types_without_attributes(
        merged_nodes if merged_nodes is not None else bgp_nodes,
        merged_edges if merged_edges is not None else bgp_edges,
        out,
    )

    # 5) Comparación de tamaño entre topologías
    plot_topology_size_comparison(bgp_nodes, bgp_edges, ripe_nodes, ripe_edges, merged_nodes, merged_edges, out)

    # 6) Solapamiento entre fuentes
    plot_source_overlap(bgp_nodes, ripe_nodes, bgp_edges, ripe_edges, out)

    # 7) Distribución de grado
    plot_degree_distribution(bgp_nodes, bgp_edges, ripe_nodes, ripe_edges, merged_nodes, merged_edges, out)

    # 8) Top 10 ASNs chilenos por relevancia
    nodes_for_top = merged_nodes if merged_nodes is not None else bgp_nodes
    edges_for_top = merged_edges if merged_edges is not None else bgp_edges
    plot_top10_chilean_asns(nodes_for_top, edges_for_top, out, metric=top_asn_metric)

    # 9) Top 10 enlaces AS-AS por frecuencia
    plot_top10_links_frequency(merged_edges, merged_nodes, out)

    # 10) Grafo simplificado filtrado de Chile
    plot_simplified_chile_graph(
        merged_nodes,
        merged_edges,
        bgp_nodes,
        ripe_nodes,
        out,
        top_n=top_n_graph,
        color_mode=graph_color_mode,
        export_pyvis_html=True,
    )

    print("Proceso finalizado. Revisa la carpeta output_figures/ para los PNG generados.")


main(
    base_data_dir="data/csv",
    output_dir="output_figures",
    ripe_targets_csv=None,
    ripe_probes_csv=None,
    top_asn_metric="degree_total",  # degree_total | path_occurrences | betweenness
    top_n_graph=80,
    graph_color_mode="source",       # source | has_peeringdb
)


[ADVERTENCIA] No existe BGP nodes: data/csv/bgp/nodes.csv
[ADVERTENCIA] No existe BGP edges: data/csv/bgp/edges.csv
[ADVERTENCIA] No existe RIPE nodes: data/csv/ripe_atlas/nodes.csv
[ADVERTENCIA] No existe RIPE edges: data/csv/ripe_atlas/edges.csv
[ADVERTENCIA] No existe Merged nodes: data/csv/merged/nodes.csv
[ADVERTENCIA] No existe Merged edges: data/csv/merged/edges.csv
[ADVERTENCIA] Gráfico 1 omitido: no hay CSV de targets RIPE Atlas.
[ADVERTENCIA] Gráfico 2 omitido: no hay CSV de probes RIPE Atlas.
[ADVERTENCIA] Gráfico 3 omitido: no hay nodos para cobertura PeeringDB.
[ADVERTENCIA] Gráfico 4 omitido: no hay nodos.
Proceso finalizado. Revisa la carpeta output_figures/ para los PNG generados.
